<a href="https://colab.research.google.com/github/JorgeMato43/Cell-Type-Classifier/blob/main/Fine_Tuning_paper_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/sartorius-research/LIVECell.git
%cd LIVECell

!pip install pycocotools
!pip install torch torchvision pycocotools opencv-python tqdm
!pip install kaggle

Cloning into 'LIVECell'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 202 (delta 0), reused 0 (delta 0), pack-reused 201 (from 1)
Receiving objects: 100% (202/202), 62.22 KiB | 2.59 MiB/s, done.
Resolving deltas: 100% (94/94), done.
/content/LIVECell


In [3]:
# Make folders
import os

def make_dir(base):

  ROOT = base
  IMG_DIR = os.path.join(ROOT, "images")
  ANN_DIR = os.path.join(ROOT, "annotations")

  os.makedirs(IMG_DIR, exist_ok=True)
  os.makedirs(ANN_DIR, exist_ok=True)

  print(ROOT)
  return ANN_DIR, IMG_DIR

In [4]:
ROOT = "/content/livecell"
ANN_DIR, IMG_DIR = make_dir(ROOT)

/content/livecell


In [5]:
# Download images and annotation files into corresponding folders
import os
import urllib.request

def download_livecell_data(ROOT, ANN_DIR):

  files_to_download = {
      "images_zip": (
          "https://livecell-dataset.s3.eu-central-1.amazonaws.com/LIVECell_dataset_2021/images.zip",
          os.path.join(ROOT, "images.zip")
      ),
      "train_json": (
          "https://livecell-dataset.s3.eu-central-1.amazonaws.com/LIVECell_dataset_2021/annotations/LIVECell/livecell_coco_train.json",
          os.path.join(ANN_DIR, "livecell_coco_train.json")
      ),
      "val_json": (
          "https://livecell-dataset.s3.eu-central-1.amazonaws.com/LIVECell_dataset_2021/annotations/LIVECell/livecell_coco_val.json",
          os.path.join(ANN_DIR, "livecell_coco_val.json")
      ),
      "test_json": (
          "https://livecell-dataset.s3.eu-central-1.amazonaws.com/LIVECell_dataset_2021/annotations/LIVECell/livecell_coco_test.json",
          os.path.join(ANN_DIR, "livecell_coco_test.json")
      ),
  }

  for name, (url, out_path) in files_to_download.items():
      if not os.path.exists(out_path):
          print(f"Downloading {name}...")
          urllib.request.urlretrieve(url, out_path)
      else:
          print(f"Already exists: {out_path}")

  print("Done.")

In [6]:
download_livecell_data(ROOT, ANN_DIR=ANN_DIR)

Done.


In [7]:
# Unzip images
import zipfile

def unzip_images(ROOT, IMG_DIR):
  zip_path = os.path.join(ROOT, "images.zip")

  with zipfile.ZipFile(zip_path, "r") as zf:
      zf.extractall(IMG_DIR)

  print("Unzipped to:", IMG_DIR)

In [8]:
unzip_images(ROOT, IMG_DIR=IMG_DIR)

Unzipped to: /content/livecell/images


In [9]:
# Cropping training and validation images to get individual cells
import os
import cv2
import numpy as np
from pycocotools.coco import COCO
from tqdm import tqdm


def crop_cell(ann_file, img_dir, out_dir):

  os.makedirs(out_dir, exist_ok=True)

  coco = COCO(ann_file)

  for img_id in tqdm(coco.getImgIds()):
      img_info = coco.loadImgs(img_id)[0]
      img_path = os.path.join(img_dir, img_info['file_name'])

      img = cv2.imread(img_path)
      if img is None:
          continue

      # label mapping derived from filename
      label = img_info['file_name'].split('_')[0]

      ann_ids = coco.getAnnIds(imgIds=img_id)
      anns = coco.loadAnns(ann_ids)

      for ann in anns:
          mask = coco.annToMask(ann)

          x, y, w, h = map(int, ann['bbox'])

          crop = img[y:y+h, x:x+w]
          mask_crop = mask[y:y+h, x:x+w]

          if crop.shape[0] < 10 or crop.shape[1] < 10:
              continue

          crop = crop * mask_crop[:, :, None]

          # resize to CNN-friendly size
          crop = cv2.resize(crop, (224, 224))

          class_dir = os.path.join(out_dir, label)
          os.makedirs(class_dir, exist_ok=True)

          save_path = os.path.join(class_dir, f"{img_id}_{ann['id']}.png")
          cv2.imwrite(save_path, crop)

In [10]:
ann_file = "/content/livecell/annotations/livecell_coco_train.json"
img_dir = "/content/livecell/images/images/livecell_train_val_images"
out_dir = "/content/cell_crops/"

crop_cell(ann_file, img_dir, out_dir)

loading annotations into memory...
Done (t=12.51s)
creating index...
index created!


100%|██████████| 3253/3253 [31:57<00:00,  1.70it/s]


In [11]:
# Validation/Test split
import shutil
import random


def train_val_test_data_dir_split(data_dir,
                                  test_data_dir=None,
                                  val_dir=None,
                                  train_dir=None,
                                  test_dir=None):
  '''
  Takes the data directory and the desired test, train,
  and validation data directories, and splits the data into said directories.
  '''

  os.makedirs(train_dir, exist_ok=True)
  os.makedirs(val_dir, exist_ok=True)
  os.makedirs(test_dir, exist_ok=True)

  if train_dir is not None and val_dir is not None:

    for cls in os.listdir(data_dir):
        imgs = os.listdir(os.path.join(data_dir, cls))
        random.shuffle(imgs)

        split = int(0.8 * len(imgs))

        for i, img in enumerate(imgs):
            src = os.path.join(data_dir, cls, img)

            if i < split:
                dst = os.path.join(train_dir, cls)
            else:
                dst = os.path.join(val_dir, cls)

            os.makedirs(dst, exist_ok=True)
            shutil.copy(src, os.path.join(dst, img))

  if test_dir is not None:
    for cls in os.listdir(data_dir):
      imgs = os.listdir(os.path.join(data_dir, cls))
      random.shuffle(imgs)

      for i, img in enumerate(imgs):
          src = os.path.join(data_dir, cls, img)

          dst = os.path.join(test_dir, cls)

          os.makedirs(dst, exist_ok=True)
          shutil.copy(src, os.path.join(dst, img))


In [ ]:
test_ann_file = "/content/livecell/annotations/livecell_coco_test.json"
test_img_dir = "/content/livecell/images/images/livecell_test_images"
test_out_dir = "/content/test_cell_crops/"

crop_cell(test_ann_file, test_img_dir, test_out_dir)

loading annotations into memory...
Done (t=6.98s)
creating index...
index created!


 29%|██▊       | 448/1564 [04:25<21:49,  1.17s/it]

In [ ]:
data_dir = "/content/cell_crops"
train_dir = "/content/data/train"
test_dir = "/content/data/test"
val_dir = "/content/data/val"
test_data_dir = "/content/test_cell_crops"

train_val_test_data_dir_split(data_dir,
                          test_data_dir,
                          train_dir,
                          val_dir,
                          test_dir)


In [ ]:
# Pytorch Dataset and DataLoader
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_out_dir, transform=transform)
num_classes = len(train_dataset.classes)

# Selecting a smaller set to train:
from torch.utils.data import Subset
import random

indices = random.sample(range(len(train_dataset)), 5000)
train_dataset = Subset(train_dataset, indices)
val_indices = random.sample(range(len(val_dataset)), 500)
val_dataset = Subset(val_dataset, val_indices)
test_indices = random.sample(range(len(test_dataset)), 1000)
test_dataset = Subset(test_dataset, test_indices)




In [ ]:
# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
from matplotlib import pyplot as plt

# Helper function to visualize performance during training
def plot_training_curves(train_losses, val_accuracies):

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(train_losses)
    ax1.set_title('Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.grid(True)

    ax2.plot(val_accuracies)
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
def save_checkpoint(model, optimizer, epoch, path="models/checkpoint.pth"):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, path)

In [ ]:
# Defining training function:
from torch.optim import SGD
from time import time
import numpy as np

def train_model(model, lr=1e-4, min_delta=0.1, patience=2, epochs=5,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=True):
  #Checking gpu access
  if torch.cuda.is_available():
    gpu = torch.device("cuda")
  elif torch.backends.mps.is_available():
    gpu = torch.device("mps")
  else:
    gpu = torch.device("cpu") # Fallback to CPU if no GPU available

  model = model.to(gpu)
  loss = nn.CrossEntropyLoss()
  optimizer = SGD(model.parameters(), lr=lr)
  epochs = epochs
  loss_list = []
  accuracy_list = []
  start_time = time()
  epoch_times = []
  counter = 0
  min_delta = min_delta
  patience = patience
  best_accuracy = 0

  if training:
    for epoch in range(epochs):
      train_loss = 0
      correct_preds = 0
      epoch_start = time()
      for (inputs, label) in training_dl:
        #Transfer to gpu
        inputs = inputs.to(gpu)
        label = label.to(gpu)
        #Forward Propagation
        outputs = model(inputs)
        epoch_loss = loss(outputs, label)
        train_loss += epoch_loss.item()

        #Backward propagation
        epoch_loss.backward()

        #Gradient Step
        optimizer.step()
        optimizer.zero_grad()

      loss_list.append(train_loss)
      #Validation
      with torch.no_grad():
        for (inputs, label) in val_dl:
          inputs = inputs.to(gpu)
          label = label.to(gpu)
          outputs = model(inputs)
          correct_preds += (torch.argmax(outputs, dim=1) == label).sum().item()

      accuracy = correct_preds / len(val_loader.dataset)
      accuracy_list.append(accuracy)
      epoch_end = time()
      epoch_times.append(epoch_end-epoch_start)
      print(f'epoch: {epoch+1}, training loss: {train_loss}, accuracy:{accuracy}')

      #Early stopping
      if accuracy > best_accuracy:
        best_accuracy = accuracy
      if accuracy < best_accuracy - min_delta:
        counter += 1
        if counter >= patience:
          print(f'Early stopping at epoch {epoch+1}')
          break
      else:
        counter = 0

    end_time = time()
    total_training_time = end_time - start_time
    avg_time_epoch = np.mean(epoch_times)
    print(f'Total trianing time: {total_training_time}')
    print(f'Average time per epoch: {avg_time_epoch}')
    save_checkpoint(model, optimizer, epoch, path="models/checkpoint.pth")
    return loss_list, accuracy_list, avg_time_epoch
  else:
    with torch.no_grad():
      correct_preds = 0
      for (inputs, label) in test_dl:
        inputs = inputs.to(gpu)
        label = label.to(gpu)
        outputs = model(inputs)
        correct_preds += (torch.argmax(outputs, dim=1) == label).sum().item()

    accuracy = correct_preds / len(test_dl.dataset)
    print(f'accuracy:{accuracy}')


## Training ResNet on Cropped Cell images

#### Baseline (Untrained) performance

In [ ]:
# Pulling torchvision ResNet
import torchvision.models as models
import torch.nn as nn

frozen_model = models.resnet18(pretrained=True)
for parameter in frozen_model.parameters():
    parameter.requires_grad = False

frozen_model.fc = nn.Linear(frozen_model.fc.in_features, num_classes)

frozen_model = frozen_model.cuda()

In [ ]:
untrained_stats = train_model(frozen_model, lr=5e-3, min_delta=0.1, patience=2, epochs=20,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=False)

#### Training the model with frozen parameters

In [ ]:
training_stats = train_model(frozen_model, lr=5e-3, min_delta=0.1, patience=2, epochs=20,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=True)

In [ ]:
plot_training_curves(training_stats[0], training_stats[1])

#### Testing on unseen data

In [ ]:
testing_stats = train_model(frozen_model, lr=5e-3, min_delta=0.1, patience=2, epochs=20,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=False)

### Trying model without frozen parameters

In [ ]:
# Pulling torchvision ResNet and leaving parameters unfrozen
unfrozen_model = models.resnet18(pretrained=True)

unfrozen_model.fc = nn.Linear(unfrozen_model.fc.in_features, num_classes)

unfrozen_model = unfrozen_model.cuda()

#### Model Baseline (Untrained)

In [ ]:
unfrozen_baseline_stats = train_model(unfrozen_model, lr=5e-3, min_delta=0.1, patience=2, epochs=20,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=False)

#### Training model with unfrozen parameters

In [ ]:
# Training model with unfrozen parameters
unfrozen_training_stats = train_model(unfrozen_model, lr=5e-3, min_delta=0.1, patience=2, epochs=20,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=True)

In [ ]:
plot_training_curves(unfrozen_training_stats[0], unfrozen_training_stats[1])

#### Testing on unseen data

In [ ]:
testing_unfrozen_stats = train_model(unfrozen_model, lr=5e-3, min_delta=0.1, patience=2, epochs=20,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=False)

### Ablation Study for learning rate in model with Frozen and Unfrozen parameters

#### Helper function to visualize training stats during the ablation study

In [ ]:
from matplotlib import pyplot as plt

# Helper function to visualize performance during training
def plot_ablation_training_curves(train_losses, val_accuracies,
                                  learning_rates, avg_time_epoch):
    for i in range(len(learning_rates)):

      fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

      ax1.plot(train_losses[i])
      ax1.set_title(f'Training Loss, lr = {learning_rates[i]}')
      ax1.set_xlabel('Epoch')
      ax1.set_ylabel('Loss')
      ax1.grid(True)

      ax2.plot(val_accuracies[i])
      ax2.set_title(f'Accuracy, lf = {learning_rates[i]}')
      ax2.set_xlabel('Epoch')
      ax2.set_ylabel('Accuracy')
      ax2.grid(True)

      plt.tight_layout()
      plt.show()

#### Frozen Parameters Model

In [ ]:
# Ablation study for learning rate
lrs = [1e-2, 1e-3, 1e-4, 1e-5]
losses = []
accuracies = []
avg_time_epochs = []
for lr in lrs:
  frozen_model = models.resnet18(pretrained=True)
  for parameter in frozen_model.parameters():
      parameter.requires_grad = False

  frozen_model.fc = nn.Linear(frozen_model.fc.in_features, num_classes)

  frozen_model = frozen_model.cuda()
  frozen_model_stats = train_model(frozen_model, lr=lr, min_delta=0.1, patience=2, epochs=20,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=True)
  losses.append(frozen_model_stats[0])
  accuracies.append(frozen_model_stats[1])
  avg_time_epochs.append(frozen_model_stats[2])


In [ ]:
plot_ablation_training_curves(losses, accuracies, lrs, avg_time_epochs)

#### Training one more time with the preferred learning rate

In [ ]:
frozen_model = models.resnet18(pretrained=True)
for parameter in frozen_model.parameters():
    parameter.requires_grad = False

frozen_model.fc = nn.Linear(frozen_model.fc.in_features, num_classes)

frozen_model = frozen_model.cuda()
frozen_model_stats = train_model(frozen_model, lr=0.0005, min_delta=0.1, patience=2,
                                 epochs=25,
                                 training_dl=train_loader, val_dl=val_loader,
                                 test_dl=test_loader, training=True)

In [ ]:
plot_training_curves(frozen_model_stats[0], frozen_model_stats[1])

#### Unfrozen Parameters Model

In [ ]:
# Ablation study for learning rate
lrs = [1e-2, 1e-3, 1e-4, 1e-5]
losses = []
accuracies = []
avg_time_epochs = []
for lr in lrs:
  unfrozen_model = models.resnet18(pretrained=True)

  unfrozen_model.fc = nn.Linear(unfrozen_model.fc.in_features, num_classes)

  unfrozen_model = unfrozen_model.cuda()
  unfrozen_model_stats = train_model(unfrozen_model, lr=lr, min_delta=0.1, patience=2, epochs=20,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=True)
  losses.append(unfrozen_model_stats[0])
  accuracies.append(unfrozen_model_stats[1])
  avg_time_epochs.append(unfrozen_model_stats[2])

In [ ]:
plot_ablation_training_curves(losses, accuracies, lrs, avg_time_epochs)

#### Training with preferred parameters

In [ ]:
unfrozen_model = models.resnet18(pretrained=True)

unfrozen_model.fc = nn.Linear(unfrozen_model.fc.in_features, num_classes)

unfrozen_model = unfrozen_model.cuda()
unfrozen_model_stats = train_model(unfrozen_model, lr=0.0005, min_delta=0.1, patience=2, epochs=25,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=True)

In [ ]:
plot_training_curves(unfrozen_model_stats[0], unfrozen_model_stats[1])

### Training on 10,000 cells with preferred parameters

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_out_dir, transform=transform)
num_classes = len(train_dataset.classes)

# Selecting a smaller set to train:
from torch.utils.data import Subset
import random

indices = random.sample(range(len(train_dataset)), 10000)
train_dataset = Subset(train_dataset, indices)
val_indices = random.sample(range(len(val_dataset)), 2000)
val_dataset = Subset(val_dataset, val_indices)
test_indices = random.sample(range(len(test_dataset)), 3000)
test_dataset = Subset(test_dataset, test_indices)

In [ ]:
# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
# Testing the untrained model
unfrozen_model = models.resnet18(pretrained=True)
unfrozen_model.fc = nn.Linear(unfrozen_model.fc.in_features, num_classes)


unfrozen_baseline_stats = train_model(unfrozen_model, lr=5e-3, min_delta=0.1, patience=2, epochs=20,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=False)

In [ ]:
#Training with preferred lr and 10,000 samples
unfrozen_training_stats = train_model(unfrozen_model, lr=0.001, min_delta=0.1, patience=2, epochs=20,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=True)

In [ ]:
plot_training_curves(unfrozen_training_stats[0], unfrozen_training_stats[1])

### Training with 100,000 samples

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_out_dir, transform=transform)
num_classes = len(train_dataset.classes)

# Selecting a smaller set to train:
from torch.utils.data import Subset
import random

indices = random.sample(range(len(train_dataset)), 100000)
train_dataset = Subset(train_dataset, indices)
val_indices = random.sample(range(len(val_dataset)), 20000)
val_dataset = Subset(val_dataset, val_indices)
test_indices = random.sample(range(len(test_dataset)), 30000)
test_dataset = Subset(test_dataset, test_indices)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
unfrozen_model = models.resnet18(pretrained=True)
unfrozen_model.fc = nn.Linear(unfrozen_model.fc.in_features, num_classes)


unfrozen_baseline_stats = train_model(unfrozen_model, lr=0.001, min_delta=0.1, patience=2, epochs=20,
                    training_dl=train_loader, val_dl=val_loader,
                    test_dl=test_loader, training=True)

In [ ]:
plot_training_curves(unfrozen_model_stats[0],unfrozen_model_stats[1])